# N077 · Trie的前缀状态与词典查询

**目标：** 把共享前缀编码成树路径并区分词尾。

**先修：** N007, N009, N069。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 字符边；终止标记；插入/搜索/前缀；字符集与空间。

## 从问题到算法

Trie节点表示一个前缀状态，不是单个完整单词。沿字符边走到某节点说明前缀存在，但只有terminal标记才能说明单词在字典中。替换词根时沿单词逐字符走，第一次遇到terminal就得到最短根，不需要枚举全部字典词。

## 最小实现

**本章接口：** `Trie.insert(word)`、`Trie.search(word)`、`Trie.startsWith(prefix)`、`replace_words(dictionary, sentence)`

In [1]:
class TrieNode:
    def __init__(self): self.children={}; self.terminal=False

class Trie:
    def __init__(self): self.root=TrieNode()
    def insert(self,word):
        node=self.root
        for c in word:
            if c not in node.children: node.children[c]=TrieNode()
            node=node.children[c]
        node.terminal=True
    def _walk(self,text):
        node=self.root
        for c in text:
            if c not in node.children: return None
            node=node.children[c]
        return node
    def search(self,word):
        node=self._walk(word); return node is not None and node.terminal
    def startsWith(self,prefix): return self._walk(prefix) is not None

def replace_words(dictionary,sentence):
    trie=Trie()
    for word in dictionary:
        if not word: raise ValueError('replacement roots must be nonempty')
        trie.insert(word)
    def replace(word):
        node=trie.root
        for i,c in enumerate(word):
            if c not in node.children: return word
            node=node.children[c]
            if node.terminal: return word[:i+1]
        return word
    return ' '.join(replace(word) for word in sentence.split())

## 正确性、前提与复杂度

每条根到节点路径唯一编码一个前缀，terminal独立记录是否有词在此结束。第一次遇到terminal对应沿该单词的最短字典前缀。

**代价：** 长度L的插入/查询平均O(L)；总节点数至多所有插入单词长度之和加1。替换总扫描字符数线性。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

trie=Trie()
for word in ['app','apple','apt']: trie.insert(word)
rows=[]; stack=[('',trie.root)]
while stack:
    prefix,node=stack.pop(); rows.append((prefix or 'ε',node.terminal,sorted(node.children)))
    for c,child in sorted(node.children.items(),reverse=True): stack.append((prefix+c,child))
show_table(['前缀状态','完整单词','出边字符'],rows)

前缀状态,完整单词,出边字符
ε,False,['a']
a,False,['p']
ap,False,"['p', 't']"
app,True,['l']
appl,False,['e']
apple,True,[]
apt,True,[]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
trie=Trie(); trie.insert('apple')
assert trie.search('apple') and not trie.search('app') and trie.startsWith('app')
trie.insert('app'); trie.insert('app'); assert trie.search('app')
assert trie.startsWith('') and not trie.search('')
trie.insert(''); assert trie.search('')
assert replace_words(['cat','bat','rat'],'the cattle was rattled by the battery')=='the cat was rat by the bat'
print('N077: 所有本章断言通过')

N077: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 支持删除词并保持其他共享前缀。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较哈希完整词查询与前缀查询。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 208. Implement Trie (Prefix Tree)（canonical）
- 648. Replace Words（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。